# Practical Application III: Comparing Classifiers

**Overview**: In this practical application, your goal is to compare the performance of the classifiers we encountered in this section, namely K Nearest Neighbor, Logistic Regression, Decision Trees, and Support Vector Machines.  We will utilize a dataset related to marketing bank products over the telephone.



### Getting Started

Our dataset comes from the UCI Machine Learning repository [link](https://archive.ics.uci.edu/ml/datasets/bank+marketing).  The data is from a Portugese banking institution and is a collection of the results of multiple marketing campaigns.  We will make use of the article accompanying the dataset [here](CRISP-DM-BANK.pdf) for more information on the data and features.



### Problem 1: Understanding the Data

To gain a better understanding of the data, please read the information provided in the UCI link above, and examine the **Materials and Methods** section of the paper.  How many marketing campaigns does this data represent?

### Problem 2: Read in the Data

Use pandas to read in the dataset `bank-additional-full.csv` and assign to a meaningful variable name.

In [ ]:
import pandas as pd
# Additional imports for analysis and modeling
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, roc_curve, average_precision_score
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
import time
sns.set_theme(style='whitegrid')
# Runtime controls
FAST_RUN = True  # set to False for full GridSearchCV
N_FOLDS = 3 if FAST_RUN else 5

In [8]:
df = pd.read_csv('data/bank-additional-full.csv', sep=';')

In [9]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


### Problem 3: Understanding the Features


Examine the data description below, and determine if any of the features are missing values or need to be coerced to a different data type.


```
Input variables:
# bank client data:
1 - age (numeric)
2 - job : type of job (categorical: 'admin.','blue-collar','entrepreneur','housemaid','management','retired','self-employed','services','student','technician','unemployed','unknown')
3 - marital : marital status (categorical: 'divorced','married','single','unknown'; note: 'divorced' means divorced or widowed)
4 - education (categorical: 'basic.4y','basic.6y','basic.9y','high.school','illiterate','professional.course','university.degree','unknown')
5 - default: has credit in default? (categorical: 'no','yes','unknown')
6 - housing: has housing loan? (categorical: 'no','yes','unknown')
7 - loan: has personal loan? (categorical: 'no','yes','unknown')
# related with the last contact of the current campaign:
8 - contact: contact communication type (categorical: 'cellular','telephone')
9 - month: last contact month of year (categorical: 'jan', 'feb', 'mar', ..., 'nov', 'dec')
10 - day_of_week: last contact day of the week (categorical: 'mon','tue','wed','thu','fri')
11 - duration: last contact duration, in seconds (numeric). Important note: this attribute highly affects the output target (e.g., if duration=0 then y='no'). Yet, the duration is not known before a call is performed. Also, after the end of the call y is obviously known. Thus, this input should only be included for benchmark purposes and should be discarded if the intention is to have a realistic predictive model.
# other attributes:

12 - campaign: number of contacts performed during this campaign and for this client (numeric, includes last contact)
13 - pdays: number of days that passed by after the client was last contacted from a previous campaign (numeric; 999 means client was not previously contacted)
14 - previous: number of contacts performed before this campaign and for this client (numeric)
15 - poutcome: outcome of the previous marketing campaign (categorical: 'failure','nonexistent','success')
# social and economic context attributes
16 - emp.var.rate: employment variation rate - quarterly indicator (numeric)
17 - cons.price.idx: consumer price index - monthly indicator (numeric)
18 - cons.conf.idx: consumer confidence index - monthly indicator (numeric)
19 - euribor3m: euribor 3 month rate - daily indicator (numeric)
20 - nr.employed: number of employees - quarterly indicator (numeric)

Output variable (desired target):
21 - y - has the client subscribed a term deposit? (binary: 'yes','no')
```



In [ ]:
# Basic overview and class balance
print('Shape:', df.shape)
print('Class counts:\n', df.y.value_counts())
print('Nulls top 5:\n', df.isna().sum().sort_values(ascending=False).head())

### Problem 4: Understanding the Task

After examining the description and data, your goal now is to clearly state the *Business Objective* of the task.  State the objective below.

### Exploratory Visualizations


In [ ]:
import os
os.makedirs('figs', exist_ok=True)

# 1) Class balance
plt.figure(figsize=(4,3))
sns.countplot(data=df, x='y', order=['no','yes'], palette='pastel')
plt.title('Class Balance (y)')
plt.xlabel('Subscribed to Term Deposit')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('figs/class_balance.png', dpi=150)
plt.close()

# 2) Categorical counts (job, marital, education)
cat_feats = ['job', 'marital', 'education']
fig, axes = plt.subplots(1, 3, figsize=(15,4))
for ax, col in zip(axes, cat_feats):
    order = df[col].value_counts().index[:10] if col == 'job' else None
    sns.countplot(data=df, x=col, hue='y', ax=ax, order=order, palette='Set2')
    ax.set_title(f'{col.title()} by Outcome')
    ax.set_xlabel(col.title())
    ax.set_ylabel('Count')
    for label in ax.get_xticklabels():
        label.set_rotation(45)
        label.set_horizontalalignment('right')
plt.tight_layout()
fig.savefig('figs/categorical_counts.png', dpi=150)
plt.close(fig)

# 3) Numeric distributions (age, campaign, euribor3m)
num_feats = [c for c in ['age','campaign','euribor3m'] if c in df.columns]
fig, axes = plt.subplots(1, len(num_feats), figsize=(5*len(num_feats),4))
if len(num_feats) == 1:
    axes = [axes]
for ax, col in zip(axes, num_feats):
    sns.histplot(data=df, x=col, hue='y', bins=30, kde=True, element='step', stat='density', common_norm=False, ax=ax)
    ax.set_title(f'Distribution of {col}')
plt.tight_layout()
fig.savefig('figs/numeric_distributions.png', dpi=150)
plt.close(fig)

# 4) Boxenplot for euribor3m by outcome (if available)
if 'euribor3m' in df.columns:
    plt.figure(figsize=(4.5,4))
    sns.boxenplot(data=df, x='y', y='euribor3m', palette='pastel')
    plt.title('euribor3m by Outcome')
    plt.xlabel('Outcome')
    plt.ylabel('euribor3m')
    plt.tight_layout()
    plt.savefig('figs/euribor_boxen.png', dpi=150)
    plt.close()

# 5) Correlation heatmap (selected numeric subset)
subset_numeric = [c for c in ['age','campaign','pdays','previous','emp.var.rate','cons.price.idx','cons.conf.idx','euribor3m','nr.employed'] if c in df.columns]
if subset_numeric:
    corr = df[subset_numeric].corr()
    plt.figure(figsize=(7,5))
    sns.heatmap(corr, cmap='vlag', center=0, annot=False)
    plt.title('Correlation Heatmap (Selected Numeric Features)')
    plt.tight_layout()
    plt.savefig('figs/corr_heatmap.png', dpi=150)
    plt.close()
df.info()

### Problem 5: Engineering Features

Now that you understand your business objective, we will build a basic model to get started.  Before we can do this, we must work to encode the data.  Using just the bank information features, prepare the features and target column for modeling with appropriate encoding and transformations.

In [ ]:
# Encode target and set up features

df = df.copy()
df['target'] = (df['y'] == 'yes').astype(int)

# Define feature lists
all_features = [c for c in df.columns if c not in ['y', 'target']]
realistic_features = [c for c in all_features if c != 'duration']  # exclude duration to avoid leakage

# Identify column types

numeric_cols = df[realistic_features].select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = [c for c in realistic_features if c not in numeric_cols]

# Preprocessor
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

X = df[realistic_features]
y = df['target']
# Split after defining X, y
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Baseline with DummyClassifier
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
base_pred = baseline.predict(X_test)
print('Baseline accuracy (most frequent):', accuracy_score(y_test, base_pred))


In [ ]:
# Build default model pipelines
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, n_jobs=None),
    'KNN': KNeighborsClassifier(),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'SVC': SVC(probability=True, random_state=42)
}

results = []
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

for name, clf in models.items():
    pipe = Pipeline(steps=[('prep', preprocessor), ('clf', clf)])
    t0 = time.perf_counter()
    pipe.fit(X_train, y_train)
    t1 = time.perf_counter()
    # CV on training
    cv_auc = cross_val_score(pipe, X_train, y_train, cv=skf, scoring='roc_auc').mean()
    # Test metrics
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1] if hasattr(pipe.named_steps['clf'], 'predict_proba') else None
    test_acc = accuracy_score(y_test, y_pred)
    test_f1 = f1_score(y_test, y_pred)
    test_auc = roc_auc_score(y_test, y_proba) if y_proba is not None else float('nan')
    results.append({
        'Model': name,
        'Train Time (s)': round(t1 - t0, 3),
        'CV ROC AUC (train)': round(cv_auc, 3),
        'Test Acc': round(test_acc, 3),
        'Test F1': round(test_f1, 3),
        'Test ROC AUC': round(test_auc, 3) if not np.isnan(test_auc) else 'N/A'
    })

# Optionally skip GridSearch in FAST_RUN mode
if not FAST_RUN:
    run_grids = True
else:
    run_grids = False
res_df = pd.DataFrame(results).sort_values(by='CV ROC AUC (train)', ascending=False)
display(res_df)

# Grid Search Hyperparameters (concise grids)
param_grids = {
    'LogisticRegression': {
        'clf__C': [0.1, 1, 10],
        'clf__class_weight': [None, 'balanced']
    },
    'KNN': {
        'clf__n_neighbors': [5, 15, 31],
        'clf__weights': ['uniform', 'distance']
    },
    'DecisionTree': {
        'clf__max_depth': [None, 5, 10, 20],
        'clf__min_samples_leaf': [1, 5, 10]
    },
    'SVC': {
        'clf__C': [0.5, 1, 4],
        'clf__gamma': ['scale', 0.1, 0.01],
        'clf__class_weight': [None, 'balanced']
    }
}

if run_grids:
    gs_rows = []
    for name, clf in models.items():
        pipe = Pipeline(steps=[('prep', preprocessor), ('clf', clf)])
        grid = param_grids[name]
        gs = GridSearchCV(pipe, grid, cv=skf, scoring='roc_auc', n_jobs=-1, refit=True, verbose=0)
        t0 = time.perf_counter()
        gs.fit(X_train, y_train)
        t1 = time.perf_counter()
        best = gs.best_estimator_
        y_pred = best.predict(X_test)
        y_proba = best.predict_proba(X_test)[:, 1] if hasattr(best.named_steps['clf'], 'predict_proba') else None
        test_acc = accuracy_score(y_test, y_pred)
        test_f1 = f1_score(y_test, y_pred)
        test_auc = roc_auc_score(y_test, y_proba) if y_proba is not None else float('nan')
        gs_rows.append({
            'Model': name,
            'Best Params': gs.best_params_,
            'CV Best ROC AUC': round(gs.best_score_, 3),
            'Fit Time (s)': round(t1 - t0, 3),
            'Test Acc': round(test_acc, 3),
            'Test F1': round(test_f1, 3),
            'Test ROC AUC': round(test_auc, 3) if not np.isnan(test_auc) else 'N/A'
        })

    gs_df = pd.DataFrame(gs_rows).sort_values(by='CV Best ROC AUC', ascending=False)
    display(gs_df)
else:
    print('Skipping GridSearchCV in FAST_RUN mode. Set FAST_RUN=False to enable.')

### Problem 6: Train/Test Split

With your data prepared, split it into a train and test set.

### Problem 7: A Baseline Model

Before we build our first model, we want to establish a baseline.  What is the baseline performance that our classifier should aim to beat?

### Problem 8: A Simple Model

Use Logistic Regression to build a basic model on your data.

### Problem 9: Score the Model

What is the accuracy of your model?

### Problem 10: Model Comparisons

Now, we aim to compare the performance of the Logistic Regression model to our KNN algorithm, Decision Tree, and SVM models.  Using the default settings for each of the models, fit and score each.  Also, be sure to compare the fit time of each of the models.  Present your findings in a `DataFrame` similar to that below:

| Model | Train Time | Train Accuracy | Test Accuracy |
| ----- | ---------- | -------------  | -----------   |
|     |    |.     |.     |

### Problem 11: Improving the Model

Now that we have some basic models on the board, we want to try to improve these.  Below, we list a few things to explore in this pursuit.

- More feature engineering and exploration.  For example, should we keep the gender feature?  Why or why not?
- Hyperparameter tuning and grid search.  All of our models have additional hyperparameters to tune and explore.  For example the number of neighbors in KNN or the maximum depth of a Decision Tree.
- Adjust your performance metric

##### Questions